In [6]:
# ============================================================
# PHASE 8: MLFLOW — ALL SETUP IN ONE CELL
# No separate configuration cell needed
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, f1_score,
    precision_score, recall_score,
    confusion_matrix
)
import joblib, os, json

# ── Load Data ──────────────────────────────────────────────
X_train = pd.read_csv('../data/processed/X_train_fe.csv')
X_test  = pd.read_csv('../data/processed/X_test_fe.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()

# ── MLflow Setup ───────────────────────────────────────────
# Using default local storage — simplest option, no config needed
# MLflow will auto-create an mlruns/ folder in your project root

mlflow.set_tracking_uri('mlruns')
mlflow.set_experiment('Customer_Churn_Prediction')

print("MLflow version  :", mlflow.__version__)
print("Tracking URI    :", mlflow.get_tracking_uri())
print("Experiment      : Customer_Churn_Prediction")
print("\nData loaded:")
print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")

MlflowException: The filesystem tracking backend (e.g., './mlruns') is in maintenance mode and will not receive further updates. Please migrate to a database backend (e.g., 'sqlite:///mlflow.db') to access the latest MLflow features. The `mlflow migrate-filestore` tool migrates your existing data losslessly. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance. If the filesystem backend is required for your workflow, set `MLFLOW_ALLOW_FILE_STORE=true` to opt out of this exception.

In [14]:
import sys
import mlflow

print("Python version :", sys.version)
print("MLflow version :", mlflow.__version__)

Python version : 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
MLflow version : 3.13.0


In [13]:
# ============================================================
# STEP 1: Configure MLflow — FIXED FOR WINDOWS
# ============================================================

import os

# Get the absolute path of the mlruns folder
# This fixes the Windows relative path issue
notebook_dir = os.path.abspath('')          # current notebook directory
project_root  = os.path.dirname(notebook_dir)  # go one level up to project root
mlruns_path   = os.path.join(project_root, 'mlruns')

print("Project root :", project_root)
print("MLruns path  :", mlruns_path)

# Set tracking URI using absolute path
mlflow.set_tracking_uri(f'file:///{mlruns_path}')

# Create or connect to our experiment
experiment_name = 'Customer_Churn_Prediction'
mlflow.set_experiment('Customer_Churn_Prediction')

print(f"\nMLflow tracking URI : file:///{mlruns_path}")
print(f"Experiment name     : {experiment_name}")
print("\nTo view the MLflow UI, open a terminal and run:")
print(f"  mlflow ui --backend-store-uri file:///{mlruns_path}")
print("  Then open: http://localhost:5000")

Project root : c:\Users\nikhi\OneDrive\Desktop\customer-churn-prediction
MLruns path  : c:\Users\nikhi\OneDrive\Desktop\customer-churn-prediction\mlruns


MlflowException: The filesystem tracking backend (e.g., './mlruns') is in maintenance mode and will not receive further updates. Please migrate to a database backend (e.g., 'sqlite:///mlflow.db') to access the latest MLflow features. The `mlflow migrate-filestore` tool migrates your existing data losslessly. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance. If the filesystem backend is required for your workflow, set `MLFLOW_ALLOW_FILE_STORE=true` to opt out of this exception.

In [1]:
# ============================================================
# STEP 2: Helper function to log any model as an MLflow run
#
# Each run logs:
#   params  → hyperparameters used
#   metrics → AUC, F1, Precision, Recall
#   model   → the actual trained model artifact
#   plots   → confusion matrix as image
# ============================================================

def log_model_run(run_name, model, params, X_tr, y_tr, X_te, y_te):
    """
    Trains a model and logs everything to MLflow.
    Returns the run_id for future reference.
    """
    with mlflow.start_run(run_name=run_name) as run:

        # ── Train ──────────────────────────────────────────
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        y_prob = model.predict_proba(X_te)[:, 1]

        # ── Metrics ────────────────────────────────────────
        metrics = {
            'auc_roc'   : roc_auc_score(y_te, y_prob),
            'f1_score'  : f1_score(y_te, y_pred),
            'precision' : precision_score(y_te, y_pred),
            'recall'    : recall_score(y_te, y_pred)
        }

        # ── Log params ─────────────────────────────────────
        # params = the hyperparameters dictionary you pass in
        mlflow.log_params(params)

        # ── Log metrics ────────────────────────────────────
        mlflow.log_metrics(metrics)

        # ── Log confusion matrix as artifact ───────────────
        cm = confusion_matrix(y_te, y_pred)
        fig, ax = plt.subplots(figsize=(5, 4))
        import seaborn as sns
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['Retained','Churned'],
                    yticklabels=['Retained','Churned'], ax=ax)
        ax.set_title(f'Confusion Matrix — {run_name}')
        plt.tight_layout()
        # Save plot temporarily, then log it
        tmp_path = f'../reports/figures/cm_{run_name.replace(" ","_")}.png'
        plt.savefig(tmp_path, bbox_inches='tight')
        plt.close()
        mlflow.log_artifact(tmp_path, artifact_path='plots')

        # ── Log model ──────────────────────────────────────
        # infer_signature captures input/output schema
        signature = infer_signature(X_tr, y_prob)
        mlflow.sklearn.log_model(
            model,
            artifact_path='model',
            signature=signature,
            input_example=X_tr.iloc[:3]
        )

        run_id = run.info.run_id
        print(f"\n{'─'*50}")
        print(f"Run  : {run_name}")
        print(f"AUC  : {metrics['auc_roc']:.4f}")
        print(f"F1   : {metrics['f1_score']:.4f}")
        print(f"Prec : {metrics['precision']:.4f}")
        print(f"Rec  : {metrics['recall']:.4f}")
        print(f"ID   : {run_id}")
        return run_id, metrics

print("Helper function ready.")

Helper function ready.


In [3]:
# ============================================================
# RUN 1: Logistic Regression (Baseline)
# ============================================================
from sklearn.linear_model import LogisticRegression
lr_params = {
    'model_type'   : 'LogisticRegression',
    'max_iter'     : 1000,
    'class_weight' : 'balanced',
    'solver'       : 'lbfgs'
}

lr_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

run_id_lr, metrics_lr = log_model_run(
    run_name = 'Logistic_Regression_Baseline',
    model    = lr_model,
    params   = lr_params,
    X_tr=X_train, y_tr=y_train,
    X_te=X_test,  y_te=y_test
)

NameError: name 'X_train' is not defined

In [ ]:
# ============================================================
# RUN 2: Random Forest
# ============================================================

rf_params = {
    'model_type'       : 'RandomForest',
    'n_estimators'     : 200,
    'max_depth'        : 10,
    'min_samples_leaf' : 10,
    'class_weight'     : 'balanced'
}

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

run_id_rf, metrics_rf = log_model_run(
    run_name = 'Random_Forest',
    model    = rf_model,
    params   = rf_params,
    X_tr=X_train, y_tr=y_train,
    X_te=X_test,  y_te=y_test
)

In [ ]:
# ============================================================
# RUN 3: XGBoost Default
# ============================================================

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_weight = round(neg/pos, 2)

xgb_params_base = {
    'model_type'       : 'XGBoost',
    'n_estimators'     : 300,
    'max_depth'        : 4,
    'learning_rate'    : 0.05,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'scale_pos_weight' : scale_weight
}

xgb_base = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_weight,
    eval_metric='auc',
    random_state=42
)

run_id_xgb, metrics_xgb = log_model_run(
    run_name = 'XGBoost_Baseline',
    model    = xgb_base,
    params   = xgb_params_base,
    X_tr=X_train, y_tr=y_train,
    X_te=X_test,  y_te=y_test
)

In [ ]:
# ============================================================
# RUN 4: XGBoost Tuned — load best params from Phase 7
# ============================================================

# Load best params from Phase 7
with open('../models/final_model_metadata.json') as f:
    metadata = json.load(f)

best_params = metadata['best_params']
best_params['model_type'] = 'XGBoost_Tuned'

tuned_xgb = xgb.XGBClassifier(
    **{k: v for k, v in best_params.items() if k != 'model_type'},
    scale_pos_weight=scale_weight,
    eval_metric='auc',
    random_state=42
)

run_id_best, metrics_best = log_model_run(
    run_name = 'XGBoost_Tuned_BEST',
    model    = tuned_xgb,
    params   = best_params,
    X_tr=X_train, y_tr=y_train,
    X_te=X_test,  y_te=y_test
)

print(f"\nBest run ID saved: {run_id_best}")

In [ ]:
# ============================================================
# STEP 3: Register the best model in MLflow Model Registry
#
# Model Registry = a versioned catalogue of production models
# Allows you to promote: Staging → Production → Archived
# This is exactly how Netflix, Uber, Airbnb manage ML models
# ============================================================

model_uri = f"runs:/{run_id_best}/model"
model_name = "ChurnPredictor"

# Register the model
registered = mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)

print(f"Model registered!")
print(f"  Name   : {registered.name}")
print(f"  Version: {registered.version}")
print(f"  URI    : {model_uri}")

# Save run ID for deployment phase
with open('../models/best_run_id.txt', 'w') as f:
    f.write(run_id_best)

print(f"\nRun ID saved to models/best_run_id.txt")
print("Use this to load the model in deployment.")

In [ ]:
# ============================================================
# STEP 4: Pull all runs and compare programmatically
# ============================================================

# Search all runs in our experiment
runs_df = mlflow.search_runs(
    experiment_names=['Customer_Churn_Prediction'],
    order_by=['metrics.auc_roc DESC']
)

# Show clean comparison table
display_cols = [
    'tags.mlflow.runName',
    'metrics.auc_roc',
    'metrics.f1_score',
    'metrics.precision',
    'metrics.recall'
]

comparison = runs_df[display_cols].copy()
comparison.columns = ['Run Name', 'AUC-ROC', 'F1', 'Precision', 'Recall']
comparison = comparison.round(4)

print("EXPERIMENT COMPARISON — All Runs:")
print(comparison.to_string(index=False))

# Visual comparison
fig, ax = plt.subplots(figsize=(11, 5))
x = range(len(comparison))
width = 0.2

bars1 = ax.bar([i - width*1.5 for i in x], comparison['AUC-ROC'],
               width, label='AUC-ROC', color='#3498db', edgecolor='white')
bars2 = ax.bar([i - width*0.5 for i in x], comparison['F1'],
               width, label='F1', color='#2ecc71', edgecolor='white')
bars3 = ax.bar([i + width*0.5 for i in x], comparison['Precision'],
               width, label='Precision', color='#f39c12', edgecolor='white')
bars4 = ax.bar([i + width*1.5 for i in x], comparison['Recall'],
               width, label='Recall', color='#e74c3c', edgecolor='white')

ax.set_xticks(list(x))
ax.set_xticklabels(comparison['Run Name'], rotation=15, ha='right')
ax.set_title('MLflow Experiment Comparison — All Runs', fontweight='bold', pad=15)
ax.set_ylabel('Score')
ax.set_ylim(0.5, 1.0)
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/16_mlflow_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
print("""
╔══════════════════════════════════════════════════════════╗
║           PHASE 8 COMPLETE — MLflow TRACKING             ║
╠══════════════════════════════════════════════════════════╣
║  4 experiments logged and tracked                        ║
║  Best model registered in MLflow Model Registry          ║
║  All runs comparable in MLflow UI                        ║
║                                                          ║
║  To launch the MLflow UI:                                ║
║    → Open terminal in project root                       ║
║    → Run: mlflow ui                                      ║
║    → Open: http://localhost:5000                         ║
║                                                          ║
║  What's logged per run:                                  ║
║    ✓ Hyperparameters                                     ║
║    ✓ AUC, F1, Precision, Recall                          ║
║    ✓ Confusion matrix plot                               ║
║    ✓ Trained model artifact                              ║
║    ✓ Input/output signature                              ║
╚══════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════╗
║           PHASE 8 COMPLETE — MLflow TRACKING             ║
╠══════════════════════════════════════════════════════════╣
║  4 experiments logged and tracked                        ║
║  Best model registered in MLflow Model Registry          ║
║  All runs comparable in MLflow UI                        ║
║                                                          ║
║  To launch the MLflow UI:                                ║
║    → Open terminal in project root                       ║
║    → Run: mlflow ui                                      ║
║    → Open: http://localhost:5000                         ║
║                                                          ║
║  What's logged per run:                                  ║
║    ✓ Hyperparameters                                     ║
║    ✓ AUC, F1, Precision, Recall                          ║
║    ✓ Confusion matrix plot                               ║
║    ✓ Trained model ar